# 🔍 Regex · ⚡ Parallel Scans · 👁️ Vision · 🌊 Power/Dirac · 💾 Memory · 🔢 6-Vectors · 🔬 ASIC/Photonics · ⚡ EM · 🔢 Integer

| § | Topic |
|---|---|
| §1 | Regex — NFA/DFA theory, named groups, lookahead, compiled patterns |
| §2 | Parallel prefix scans — work/depth, Blelloch algorithm |
| §3 | Vision morphology — dilation/erosion, opening/closing, pyramid |
| §4 | Average power + Dirac delta — PSD, Parseval, impulse response |
| §5 | Memory hierarchy — cache lines, stride, spatial/temporal locality |
| §6 | 6-vectors + matrix multiply — Lorentz, FLOP counting, tiling |
| §7 | ASIC matmul + photonics — systolic array, MZI optical matmul |
| §8 | Early electromagnetism — Coulomb to Maxwell, from first principles |
| §9 | Integer arithmetic — Python big int, Montgomery, modular expo |

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import re, sys, time, math, warnings
from scipy import signal, ndimage
warnings.filterwarnings('ignore')
np.random.seed(2026)
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math
print('Regex - Parallel - Vision - Power - Memory - 6Vec - ASIC - EM - Integer  ready')

---
## §1 🔍 Regex — NFA/DFA Theory · Named Groups · Lookahead · Performance

**NFA (nondeterministic finite automaton):** Thompson construction converts a regex
to an NFA in $O(n)$ time. Each NFA state tracks a set of possible positions.

**DFA:** NFA subset construction → exponential states in worst case but $O(n)$ matching.
Python `re` uses NFA backtracking — catastrophic backtracking possible on pathological inputs.

**Named groups:** `(?P<name>...)` — accessible via `m.group('name')`.

**Lookahead/lookbehind:** `(?=...)` positive lookahead, `(?!...)` negative, `(?<=...)` lookbehind.
Zero-width — consume no characters.

**Compiled patterns:** `re.compile(pattern)` amortizes compilation over many matches.

In [ ]:
# §1 Regex — comprehensive demo

import re, timeit

# §1.1 Named groups: parsing structured data
print('§1.1 Named groups:')
LOG_PATTERN = re.compile(
    r'(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+'
    r'(?P<level>DEBUG|INFO|WARN|ERROR)\s+'
    r'\[(?P<module>[^\]]+)\]\s+'
    r'(?P<msg>.+)'
)
log_lines = [
    '2026-06-11 14:23:01 INFO  [gs_core] GS converged in 42 iterations',
    '2026-06-11 14:23:02 WARN  [beamformer] sidelobe -11.2 dB (threshold -13)',
    '2026-06-11 14:23:03 ERROR [mzi_mesh] phase drift 0.034 rad exceeding budget',
    '2026-06-11 14:23:04 DEBUG [memory] cache miss rate 23.4% on stride-8 access',
]
for line in log_lines:
    m = LOG_PATTERN.match(line)
    if m:
        print(f'  [{m.group("level"):5s}] {m.group("module"):12s} | {m.group("msg")[:50]}')

# §1.2 Lookahead / lookbehind
print('\n§1.2 Lookahead / lookbehind:')
text = 'price: $100 cost: $250 refund: $-30 fee: $5.99'
# Positive lookbehind for $ sign
amounts = re.findall(r'(?<=\$)-?\d+\.?\d*', text)
print(f'  Amounts after $: {amounts}')
# Negative lookahead: words NOT followed by colon
words   = re.findall(r'\b\w+\b(?!:)', text)
print(f'  Words not followed by colon: {words[:8]}')
# Named group + lookahead: IPv4 with port
addr_pat = re.compile(
    r'(?P<ip>(?:\d{1,3}\.){3}\d{1,3})(?::(?P<port>\d{1,5}))?'
)
addrs = ['192.168.1.1:8080', '10.0.0.5', '172.16.254.1:443']
for a in addrs:
    m = addr_pat.match(a)
    if m:
        print(f'  IP={m.group("ip"):16s} port={m.group("port") or "default"}')

# §1.3 Physics/engineering patterns
print('\n§1.3 Physics expression parser (regex):')
# Match floating point in scientific notation
sci_pat  = re.compile(r'[+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?')
# Match unit with SI prefix
unit_pat = re.compile(r'(?P<val>[+-]?\d+\.?\d*)\s*(?P<prefix>[TGMkmuunpf]?)(?P<unit>[a-zA-Z]+)')

physics_strings = [
    'lambda = 1.55e-6 m  bandwidth = 100 GHz  power = -3 dBm',
    'E_field = 1.2e5 V/m  sigma = 5.8e7 S/m  eps_r = 12.5',
    'hbar = 1.055e-34 Js  m_e = 9.109e-31 kg  c = 2.998e8 m/s',
]
for s in physics_strings:
    nums = sci_pat.findall(s)
    print(f'  Numbers: {nums}')

# §1.4 Catastrophic backtracking demo + safe alternative
print('\n§1.4 Catastrophic backtracking:')
# BAD pattern: (a+)+ on non-matching input -> exponential
bad_input = 'a' * 25 + 'b'   # just 25 chars + mismatch
safe_pat  = re.compile(r'a+b')        # atomic equivalent
bad_pat   = re.compile(r'(?:a+)+b')   # catastrophic potential

t0 = timeit.default_timer()
safe_pat.search(bad_input)
t_safe = timeit.default_timer() - t0

# Only test bad on small input to avoid hanging
t0 = timeit.default_timer()
bad_pat.search('a'*15+'b')
t_bad = timeit.default_timer() - t0

print(f'  Safe  r"a+b"    on 25a+b: {t_safe*1e6:.2f} us')
print(f'  Risky r"(?:a+)+b" on 15a+b: {t_bad*1e6:.2f} us')
print(f'  Fix: use atomic groups, possessive quantifiers, or regex2 module')

# §1.5 Tokenizer via regex (reuse from §3 of prior notebook — now benchmarked)
TOKEN_RE = re.compile(
    r'(?P<FLOAT>\d+\.\d*|\.\d+)|(?P<INT>\d+)|(?P<IDENT>[A-Za-z_]\w*)'
    r'|(?P<OP>[+\-*/^%])|(?P<LPAREN>\()|(?P<RPAREN>\))'
    r'|(?P<SKIP>[ \t\n]+)'
)
exprs = ['sin(x^2 + 3.14) * exp(-0.5*x)', '2*pi*f*L - 1/(2*pi*f*C)']
print('\n§1.5 Regex tokenizer:')
for expr in exprs:
    toks = [(m.lastgroup, m.group()) for m in TOKEN_RE.finditer(expr)
            if m.lastgroup != 'SKIP']
    print(f'  {expr[:35]:35s} -> {len(toks)} tokens: {toks[:5]}...')

# §1.6 Compiled vs interpreted benchmark
big_text = ' '.join([f'sensor_{i}=1.{i:04d}e-3' for i in range(5000)])
compiled = re.compile(r'\d+\.\d+e-\d+')
n_bench  = 50
t_comp   = timeit.timeit(lambda: compiled.findall(big_text), number=n_bench)/n_bench
t_interp = timeit.timeit(lambda: re.findall(r'\d+\.\d+e-\d+', big_text), number=n_bench)/n_bench
print(f'\n§1.6 Compiled vs interpreted on 5000-item text:')
print(f'  Compiled:   {t_comp*1000:.3f} ms/call')
print(f'  Interpreted:{t_interp*1000:.3f} ms/call  ({t_interp/t_comp:.1f}x slower)')

# ── NFA state diagram (schematic via matplotlib) ───────────────
fig, axes = plt.subplots(1,2,figsize=(14,4))
ax1 = axes[0]
ax1.axis('off')
ax1.set_title('§1 Thompson NFA for r"ab*c"\n(circles=states, arcs=transitions)', fontsize=10)
states_nfa = {0:(0.1,0.5), 1:(0.3,0.5), 2:(0.5,0.5),
              3:(0.5,0.2), 4:(0.7,0.5), 5:(0.9,0.5)}
for s,(x,y) in states_nfa.items():
    col = '#27ae60' if s==5 else '#3498db' if s==0 else 'white'
    ax1.add_patch(plt.Circle((x,y),0.07,color=col,ec='#333',lw=2,zorder=3))
    ax1.text(x,y,str(s),ha='center',va='center',fontsize=11,fontweight='bold')
transitions = [(0,1,'a'),(1,2,'eps'),(2,3,'b'),(3,2,'eps'),(2,4,'eps'),(4,5,'c')]
for (s1,s2,lbl) in transitions:
    x1,y1 = states_nfa[s1]; x2,y2 = states_nfa[s2]
    ax1.annotate('',xy=(x2,y2),xytext=(x1,y1),
                 arrowprops=dict(arrowstyle='-|>',color='#555',lw=1.8,
                                 connectionstyle='arc3,rad=0.2'))
    mx,my = (x1+x2)/2, (y1+y2)/2+0.08
    ax1.text(mx,my,lbl,fontsize=9,ha='center',color='red',fontweight='bold')
ax1.text(0.1,0.65,'start',fontsize=8,ha='center')
ax1.text(0.9,0.65,'accept',fontsize=8,ha='center')

ax2 = axes[1]
ax2.axis('off')
ax2.set_title('§1 DFA subset construction\n(exponential blowup in worst case)', fontsize=10)
# Show complexity comparison
complexities = {'NFA construction': 'O(n)', 'NFA simulation': 'O(n*m)',
                'DFA construction': 'O(2^n)','DFA matching': 'O(m)',
                'Python re (backtrack)':'O(m) avg, O(2^n) worst'}
for i,(name,cplx) in enumerate(complexities.items()):
    y_i = 0.85-i*0.16
    col = '#e74c3c' if 'worst' in cplx else '#27ae60' if 'O(m)' in cplx and 'DFA' in name else '#3498db'
    ax2.add_patch(mpatches.FancyBboxPatch((0.02,y_i-0.06),0.95,0.11,
                                           boxstyle='round,pad=0.01',fc=col,ec='white',lw=1.5,alpha=0.8))
    ax2.text(0.50,y_i,f'{name:30s}  {cplx}',ha='center',va='center',
             fontsize=9,color='white',fontweight='bold')
plt.suptitle('§1: Regex — NFA Thompson Construction · DFA · Lookahead · Compiled Performance',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §2 ⚡ Parallel Prefix Scans — Blelloch · Work/Depth · GPU Model

**Prefix scan** (inclusive): $y_i = x_0 \oplus x_1 \oplus \cdots \oplus x_i$

**Sequential:** $O(n)$ work, $O(n)$ depth (not parallelizable)

**Blelloch up-sweep/down-sweep:**
- Up-sweep (reduce): $O(n)$ work, $O(\log n)$ depth
- Down-sweep:        $O(n)$ work, $O(\log n)$ depth
- Total: $O(n)$ work, $O(\log n)$ depth — **work-optimal**

**Applications:** stream compaction, radix sort, BFS frontier, convolution (via FFT),
and importantly — **cumulative sum for attention mask** in transformer LLMs.

In [ ]:
# §2 Parallel prefix scans

import numpy as np
import time

# §2.1 Sequential scan (baseline)
def scan_sequential(arr, op=np.add):
    out = np.empty_like(arr)
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = op(out[i-1], arr[i])
    return out

# §2.2 Blelloch exclusive scan (simulated in Python, but algorithm is exact)
def blelloch_scan(arr_in, op=np.add, identity=0):
    n = len(arr_in)
    assert (n & (n-1)) == 0, 'Length must be power of 2'
    arr = arr_in.copy().astype(float)
    # Up-sweep (reduce)
    stride = 1
    while stride < n:
        for i in range(stride*2-1, n, stride*2):
            arr[i] = op(arr[i-stride], arr[i])
        stride *= 2
    # Set root to identity
    arr[-1] = identity
    # Down-sweep
    stride = n // 2
    while stride >= 1:
        for i in range(stride*2-1, n, stride*2):
            left = arr[i-stride]
            arr[i-stride] = arr[i]
            arr[i] = op(arr[i], left)
        stride //= 2
    return arr   # exclusive scan

def blelloch_inclusive(arr_in, op=np.add):
    excl = blelloch_scan(arr_in, op)
    # inclusive = shift exclusive by 1 + add original
    return excl + arr_in   # element-wise for addition

# §2.3 Parallel scan with numpy (actual parallelism via vectorized ops — Ladner-Fischer)
def ladner_fischer_scan(arr_in):
    '''Vectorized parallel prefix scan using numpy (O(n log n) work, O(log n) depth).'''
    arr = arr_in.copy().astype(float)
    n   = len(arr)
    d   = int(np.log2(n))
    # Up phase
    for k in range(d):
        stride = 2**(k+1)
        indices = np.arange(stride-1, n, stride)
        arr[indices] += arr[indices - 2**k]
    # Down phase
    arr[-1] = 0
    for k in range(d-1,-1,-1):
        stride = 2**(k+1)
        left_idx  = np.arange(stride-1, n, stride) - 2**k
        right_idx = np.arange(stride-1, n, stride)
        tmp            = arr[left_idx].copy()
        arr[left_idx]  = arr[right_idx]
        arr[right_idx] = arr[right_idx] + tmp
    return arr + arr_in   # inclusive

# §2.4 Test correctness
N = 16
x = np.arange(1, N+1, dtype=float)
expected = np.cumsum(x)

seq_out    = scan_sequential(x)
blel_out   = blelloch_inclusive(x)
lf_out     = ladner_fischer_scan(x)

print('§2 Prefix scan correctness (N=16):')
print(f'  Input:     {x.astype(int).tolist()}')
print(f'  Expected:  {expected.astype(int).tolist()}')
print(f'  Blelloch:  {blel_out.astype(int).tolist()}')
print(f'  Ladner-F:  {lf_out.astype(int).tolist()}')
print(f'  All match: {np.allclose(expected,blel_out) and np.allclose(expected,lf_out)}')

# §2.5 Work/depth analysis
print('\n§2.5 Work/depth complexity:')
for N_size in [256, 1024, 4096, 16384, 65536]:
    work_seq = N_size
    depth_seq= N_size
    work_bll = 2*N_size   # ~2n operations
    depth_bll= 2*int(np.log2(N_size))  # 2*log2(N)
    work_lf  = N_size*int(np.log2(N_size))  # n*log(n) work
    print(f'  N={N_size:6d}: seq(W={work_seq},D={depth_seq})  '
          f'Blelloch(W={work_bll},D={depth_bll})  '
          f'LF(W={work_lf},D={int(np.log2(N_size))})')

# §2.6 Performance benchmark (vectorized vs loop)
sizes = [256, 1024, 4096, 16384]
print('\n§2.6 Performance (Python simulate, us):')
print(f'  {"N":7s} {"np.cumsum":12s} {"Blelloch":12s} {"ratio"}')
for sz in sizes:
    x_b = np.random.randn(sz)
    t0  = time.perf_counter()
    for _ in range(500): np.cumsum(x_b)
    t_np = (time.perf_counter()-t0)/500*1e6

    # Pad to power of 2
    p2  = 2**int(np.ceil(np.log2(sz)))
    x_p = np.zeros(p2); x_p[:sz] = x_b
    t0  = time.perf_counter()
    for _ in range(50): ladner_fischer_scan(x_p)
    t_lf = (time.perf_counter()-t0)/50*1e6
    print(f'  {sz:7d} {t_np:12.2f} {t_lf:12.2f} {t_lf/t_np:.1f}x (overhead: Python loop)')

# §2.7 Stream compaction (key application)
print('\n§2.7 Stream compaction via prefix scan:')
data   = np.array([5,0,3,0,0,7,0,2,9,0,1,0], dtype=float)
mask   = (data > 0).astype(float)
offsets= np.cumsum(mask) - mask   # exclusive scan of mask
compacted = np.zeros(int(mask.sum()))
for i,m in enumerate(mask):
    if m:
        compacted[int(offsets[i])] = data[i]
print(f'  Input:     {data.astype(int).tolist()}')
print(f'  Mask:      {mask.astype(int).tolist()}')
print(f'  Compacted: {compacted.astype(int).tolist()}')
print(f'  GPU kernel: each thread writes to exclusive-scan offset atomically')

# Visualization
fig, axes = plt.subplots(1,3,figsize=(15,5))
# Scan comparison
x_v = np.random.randint(1,10,16).astype(float)
axes[0].plot(np.cumsum(x_v),      'o-', lw=2.5, label='Sequential numpy cumsum')
axes[0].plot(blelloch_inclusive(x_v),'s--',lw=2,label='Blelloch inclusive')
axes[0].set_title('§2 Prefix scan outputs\n(both correct)')
axes[0].legend(fontsize=9); axes[0].set_xlabel('Index')

# Work vs N
N_range = np.logspace(2,6,50)
axes[1].loglog(N_range, N_range,         'b-',  lw=2.5, label='Sequential O(N)')
axes[1].loglog(N_range, 2*N_range,       'g--', lw=2,   label='Blelloch O(N) work')
axes[1].loglog(N_range, N_range*np.log2(N_range),'r:',lw=2,label='Ladner-F O(N log N)')
axes[1].set_xlabel('N'); axes[1].set_ylabel('Work (operations)')
axes[1].set_title('§2.5 Work complexity')
axes[1].legend(fontsize=9)

# Blelloch tree diagram (schematic)
ax3 = axes[2]
ax3.axis('off')
ax3.set_title('§2 Blelloch up/down-sweep\n(N=8 example)', fontsize=10)
vals_up  = [3,1,7,0,4,1,6,3]   # example input
cum_up   = np.cumsum(vals_up)
y_levels = [0.85, 0.65, 0.45, 0.25]
x_nodes  = [np.linspace(0.05,0.95,8),
            [0.125,0.375,0.625,0.875],
            [0.25,0.75],
            [0.50]]
labels_up= [vals_up,
            [vals_up[1]+vals_up[0],vals_up[3]+vals_up[2],
             vals_up[5]+vals_up[4],vals_up[7]+vals_up[6]],
            [sum(vals_up[:4]),sum(vals_up[4:])],
            [sum(vals_up)]]
for level,(yl,xns,lbls) in enumerate(zip(y_levels,x_nodes,labels_up)):
    for xi,lbl in zip(xns,lbls):
        col = plt.cm.Blues(0.3+0.2*level)
        ax3.add_patch(plt.Circle((xi,yl),0.045,color=col,ec='#333',lw=1.5,zorder=3))
        ax3.text(xi,yl,str(lbl),ha='center',va='center',fontsize=8,fontweight='bold')
ax3.text(0.5,0.95,'Up-sweep (reduce phase)',ha='center',fontsize=9,fontweight='bold',color='#2c3e50')
ax3.text(0.5,0.05,'O(log N) depth -> O(log N) parallel time',ha='center',fontsize=8,color='gray')

plt.suptitle('§2: Parallel Prefix Scans — Blelloch · Work-Optimal · Stream Compaction',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 👁️ Vision Morphology — Max/Min Filtering · Dilation/Erosion · Pyramid

**Dilation:** $\delta_B(f)(x) = \max_{b\in B} f(x-b)$ — expands bright regions.
**Erosion:** $\varepsilon_B(f)(x) = \min_{b\in B} f(x+b)$ — shrinks bright regions.
**Opening:** $\gamma = \delta \circ \varepsilon$ (erosion then dilation) — removes small bright blobs.
**Closing:** $\phi = \varepsilon \circ \delta$ — removes small dark holes.

**Gaussian pyramid:** downsample by 2 after low-pass filtering. Scale space for
feature detection (SIFT, Harris corners) and optical flow.

**Average power** through morphological filter: preserved in opening/closing
(idempotent operators) but reduced in erosion.

In [ ]:
# §3 Vision morphology + max/min filtering

from scipy import ndimage
import numpy as np
import matplotlib.pyplot as plt

# §3.1 Create test image
N_img = 128
img   = np.zeros((N_img,N_img))
# Geometric shapes
for cx,cy,r,val in [(30,30,10,0.9),(70,50,15,0.7),(50,90,8,1.0),
                     (90,90,5,0.8),(20,80,12,0.6)]:
    rr,cc = np.ogrid[:N_img,:N_img]
    img[(rr-cy)**2+(cc-cx)**2 < r**2] = val
img += np.random.randn(N_img,N_img)*0.05   # add noise
img  = np.clip(img,0,1)

# §3.2 Morphological operations
struct_disk3 = ndimage.generate_binary_structure(2,2)   # 3x3 full connectivity
struct_disk7 = np.zeros((7,7)); rr7,cc7=np.ogrid[:7,:7]
struct_disk7[(rr7-3)**2+(cc7-3)**2<=9] = 1   # disk SE radius 3

dilated3 = ndimage.grey_dilation(img,  structure=struct_disk3.astype(bool))
eroded3  = ndimage.grey_erosion(img,   structure=struct_disk3.astype(bool))
opened3  = ndimage.grey_dilation(eroded3, structure=struct_disk3.astype(bool))
closed3  = ndimage.grey_erosion(dilated3, structure=struct_disk3.astype(bool))

# Top-hat transform: bright features on dark background
tophat   = img - opened3
blackhat = closed3 - img

# §3.3 Max and min filters (running window)
maxfilt5 = ndimage.maximum_filter(img, size=5)
minfilt5 = ndimage.minimum_filter(img, size=5)
rangefilt= maxfilt5 - minfilt5   # local range (edge detector)

print('§3 Morphological analysis:')
print(f'  Original  : mean={img.mean():.4f}  max={img.max():.4f}  min={img.min():.4f}')
print(f'  Dilated   : mean={dilated3.mean():.4f}  (brighter -> max filter)')
print(f'  Eroded    : mean={eroded3.mean():.4f}  (darker -> min filter)')
print(f'  Opened    : mean={opened3.mean():.4f}  (erosion+dilation, removes blobs)')
print(f'  Closed    : mean={closed3.mean():.4f}  (dilation+erosion, fills holes)')
print(f'  Top-hat   : sum={tophat.sum():.2f}  (bright feature extractor)')

# Idempotency check
opened_twice = ndimage.grey_dilation(
    ndimage.grey_erosion(opened3, structure=struct_disk3.astype(bool)),
    structure=struct_disk3.astype(bool))
print(f'  Opening idempotent: {np.allclose(opened3,opened_twice,atol=1e-10)}')

# §3.4 Gaussian pyramid
def build_pyramid(image, levels=4):
    pyramid = [image.copy()]
    for _ in range(levels-1):
        blurred = ndimage.gaussian_filter(pyramid[-1], sigma=1.0)
        # Downsample by 2
        pyramid.append(blurred[::2, ::2])
    return pyramid

pyramid = build_pyramid(img, levels=4)
print('\n§3.4 Gaussian pyramid:')
for i,p in enumerate(pyramid):
    power = np.mean(p**2)
    print(f'  Level {i}: shape={p.shape}  avg_power={power:.5f}')

# §3.5 Harris corner detector
def harris_corners(image, k=0.04, sigma=1.5):
    Ix = ndimage.sobel(image, axis=1)
    Iy = ndimage.sobel(image, axis=0)
    Ixx= ndimage.gaussian_filter(Ix*Ix, sigma)
    Iyy= ndimage.gaussian_filter(Iy*Iy, sigma)
    Ixy= ndimage.gaussian_filter(Ix*Iy, sigma)
    det  = Ixx*Iyy - Ixy**2
    trace= Ixx+Iyy
    R    = det - k*trace**2
    return R

R_harris = harris_corners(img)
corners  = (R_harris > R_harris.max()*0.15) & (ndimage.maximum_filter(R_harris,5)==R_harris)
cy_h,cx_h= np.where(corners)
print(f'\n§3.5 Harris corners detected: {len(cx_h)}')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3,4,figsize=(16,11))
show_data = [
    (img,      'Original + noise',    0,0),
    (dilated3, 'Dilation (3x3)',       0,1),
    (eroded3,  'Erosion (3x3)',        0,2),
    (opened3,  'Opening (E then D)',   0,3),
    (closed3,  'Closing (D then E)',   1,0),
    (tophat,   'Top-hat (img-opened)', 1,1),
    (blackhat, 'Black-hat (closed-img)',1,2),
    (rangefilt,'Local range (edges)',  1,3),
    (maxfilt5, 'Max filter 5x5',       2,0),
    (minfilt5, 'Min filter 5x5',       2,1),
]
for (data,title,r,c) in show_data:
    im = axes[r][c].imshow(data, cmap='gray', vmin=0, vmax=1)
    axes[r][c].set_title(title, fontsize=9)
    axes[r][c].axis('off')
    plt.colorbar(im, ax=axes[r][c])

# Harris corners on original
axes[2][2].imshow(img, cmap='gray')
axes[2][2].plot(cx_h, cy_h, 'r+', ms=8, mew=1.5)
axes[2][2].set_title(f'Harris corners\n({len(cx_h)} detected)', fontsize=9)
axes[2][2].axis('off')

# Gaussian pyramid
ax_pyr = axes[2][3]
ax_pyr.axis('off')
y_off  = 0
for i,p in enumerate(pyramid):
    p_show = p / p.max() if p.max() > 0 else p
    ax_pyr.imshow(p_show, cmap='hot',
                  extent=[0, p.shape[1]/N_img, y_off, y_off+p.shape[0]/N_img],
                  aspect='auto')
    y_off += p.shape[0]/N_img + 0.05
ax_pyr.set_title(f'Gaussian pyramid\n(4 levels, 2x each)', fontsize=9)

plt.suptitle('§3: Vision Morphology — Dilation/Erosion/Opening/Closing/Top-hat/Harris/Pyramid',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §4 🌊 Average Power · Dirac Delta · PSD · Parseval

**Average power** of a signal $x(t)$ over period $T$:
$$P_{avg} = \frac{1}{T}\int_0^T |x(t)|^2\,dt = \sum_n |c_n|^2 \quad\text{(Parseval)}$$

**Dirac delta** properties:
$$\int_{-\infty}^{\infty} \delta(t-t_0)\,f(t)\,dt = f(t_0)$$
$$\delta(at) = \frac{1}{|a|}\delta(t), \quad \delta(g(t)) = \sum_k \frac{\delta(t-t_k)}{|g'(t_k)|}$$
$$\mathcal{F}\{\delta(t)\} = 1, \quad \mathcal{F}\{1\} = 2\pi\delta(\omega)$$

**PSD** (Power Spectral Density): $S_{xx}(f) = |\mathcal{F}\{x\}|^2/T$

**Parseval's theorem:** $\int|x(t)|^2\,dt = \int|X(f)|^2\,df$

In [ ]:
# §4 Average power + Dirac delta + PSD + Parseval

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig

# §4.1 Dirac delta approximations
print('§4.1 Dirac delta approximations:')
t_arr = np.linspace(-3, 3, 10000)
dt    = t_arr[1]-t_arr[0]

def gaussian_delta(t, eps):
    return np.exp(-t**2/(2*eps**2)) / (eps*np.sqrt(2*np.pi))

def sinc_delta(t, N):
    return np.sinc(N*t)*N   # = sin(pi*N*t)/(pi*t)

def rect_delta(t, eps):
    return np.where(np.abs(t) < eps/2, 1/eps, 0.0)

approx = {
    'Gaussian eps=0.5':  gaussian_delta(t_arr, 0.5),
    'Gaussian eps=0.2':  gaussian_delta(t_arr, 0.2),
    'Gaussian eps=0.05': gaussian_delta(t_arr, 0.05),
    'Rect eps=0.1':      rect_delta(t_arr, 0.1),
    'Sinc N=5':          sinc_delta(t_arr, 5),
}
# Test sifting property: integral should -> f(0)
f_test = np.exp(-t_arr**2)   # f(t) = e^{-t^2}, f(0)=1
for name, delta_approx in approx.items():
    sift_val = np.trapz(delta_approx * f_test, t_arr)
    norm_val = np.trapz(delta_approx, t_arr)
    print(f'  {name:22s}: sifting integral={sift_val:.5f} (->1)  norm={norm_val:.5f} (->1)')

# §4.2 Average power for standard signals
fs     = 10000    # Hz
T_sig  = 1.0      # seconds
t_s    = np.arange(0, T_sig, 1/fs)
f0     = 50.0     # 50 Hz

signals = {
    'Sine A=1':          np.sin(2*np.pi*f0*t_s),
    'Square A=1':        np.sign(np.sin(2*np.pi*f0*t_s)),
    'Triangle A=1':      sig.sawtooth(2*np.pi*f0*t_s, 0.5),
    'Sawtooth A=1':      sig.sawtooth(2*np.pi*f0*t_s),
    'White noise std=1': np.random.randn(len(t_s)),
    'AM signal':         (1+0.5*np.sin(2*np.pi*5*t_s))*np.sin(2*np.pi*f0*t_s),
}
analytical_power = {
    'Sine A=1': 0.5, 'Square A=1': 1.0, 'Triangle A=1': 1/3,
    'Sawtooth A=1': 1/3, 'White noise std=1': 1.0, 'AM signal': 0.5*(1+0.5**2/2),
}

print('\n§4.2 Average power:')
for name, x_s in signals.items():
    P_time = np.mean(x_s**2)   # time domain
    X      = np.fft.rfft(x_s) / len(x_s)
    P_freq = 2*np.sum(np.abs(X)**2)   # Parseval (rfft)
    P_anal = analytical_power.get(name, None)
    print(f'  {name:22s}: P_time={P_time:.5f}  P_freq={P_freq:.5f}'
          + (f'  P_anal={P_anal:.5f}' if P_anal else ''))

# §4.3 Power Spectral Density
print('\n§4.3 PSD analysis:')
# Colored noise: 1/f^alpha
N_psd = 8192
white  = np.random.randn(N_psd)
f_psd  = np.fft.rfftfreq(N_psd, 1/fs)
W_psd  = np.fft.rfft(white)

# Shape noise in frequency domain
pink_W  = W_psd / np.where(f_psd>0, np.sqrt(f_psd), 1)   # 1/f
brown_W = W_psd / np.where(f_psd>0, f_psd, 1)             # 1/f^2
pink    = np.fft.irfft(pink_W)[:N_psd]
brown   = np.fft.irfft(brown_W)[:N_psd]

for name, x_n in [('White',white),('Pink (1/f)',pink),('Brown (1/f^2)',brown)]:
    f_w,Pxx = sig.welch(x_n, fs=fs, nperseg=512)
    # Fit slope on log-log
    valid = (f_w>10) & (f_w<1000)
    if valid.sum() > 5:
        slope = np.polyfit(np.log10(f_w[valid]), np.log10(Pxx[valid]+1e-20), 1)
        print(f'  {name:18s}: PSD slope = {slope[0]:.2f} (white=0, pink=-1, brown=-2)')

# §4.4 Impulse response and convolution
print('\n§4.4 Impulse response (LTI system):')
# System: RC low-pass filter, H(jw) = 1/(1 + jwRC)
RC = 1/(2*np.pi*1000)   # cutoff 1 kHz
t_ir = np.arange(0, 5*RC, 1/fs)
h_rc = (1/RC)*np.exp(-t_ir/RC)   # impulse response (IIR approximated)

# Verify: h(t) is the inverse FT of H(jw)
# H(jw) = 1/(1+jwRC) -> h(t) = (1/RC)exp(-t/RC)u(t)
f_H   = np.fft.rfftfreq(N_psd, 1/fs)
H_jw  = 1/(1 + 1j*2*np.pi*f_H*RC)
h_ifft= np.fft.irfft(H_jw)

# Apply to a test input
x_in  = np.random.randn(N_psd)
y_out = sig.lfilter([1/(RC*fs)], [1, -np.exp(-1/(RC*fs))], x_in)   # exact IIR

P_in  = np.mean(x_in**2)
P_out = np.mean(y_out**2)
# Theoretical gain: int |H(f)|^2 * S_xx(f) df / int S_xx df
H_gain= np.mean(np.abs(H_jw)**2)
print(f'  RC filter f_c=1kHz:')
print(f'  P_in={P_in:.5f}  P_out={P_out:.5f}  ratio={P_out/P_in:.5f}')
print(f'  Theoretical H_gain={H_gain:.5f}  (Parseval: matches P_out/P_in)')

# Parseval numeric verification
X_in_full = np.fft.fft(x_in)
P_time = np.mean(x_in**2)
P_freq = np.sum(np.abs(X_in_full)**2) / len(x_in)**2
print(f'\n  Parseval check: P_time={P_time:.6f}  P_freq={P_freq:.6f}  match={abs(P_time-P_freq)<1e-10}')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,8))

# Delta approximations
ax1 = axes[0][0]
for name,da in approx.items():
    if 'eps' in name:
        ax1.plot(t_arr, da, lw=2, label=name)
ax1.set_xlim(-1,1); ax1.set_ylim(-0.5,10)
ax1.set_title('§4.1 Dirac delta\nGaussian approximations')
ax1.legend(fontsize=8); ax1.set_xlabel('t')

# PSD of colored noise
ax2 = axes[0][1]
for name,xn,col in [('White',white,'gray'),('Pink',pink,'royalblue'),('Brown',brown,'brown')]:
    f_w,Pxx = sig.welch(xn,fs=fs,nperseg=1024)
    ax2.loglog(f_w[1:],Pxx[1:],color=col,lw=2,label=name)
# Reference slopes
f_ref = np.logspace(1,3,100)
ax2.loglog(f_ref, 1e4/f_ref,   'b--',lw=1.5,alpha=0.5,label='1/f ref')
ax2.loglog(f_ref, 1e8/f_ref**2,'r--',lw=1.5,alpha=0.5,label='1/f^2 ref')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('PSD')
ax2.set_title('§4.3 Power Spectral Density\nwhite/pink/brown noise')
ax2.legend(fontsize=8)

# Signals and their power
ax3 = axes[0][2]
t_show = t_s[:200]
for name,xs,col in [('Sine','royalblue','b'),('Square','red','r'),('Triangle','green','g')]:
    ax3.plot(t_show, signals[name][:200], lw=2, label=name, alpha=0.8)
ax3.set_xlabel('t (s)'); ax3.set_ylabel('Amplitude')
ax3.set_title('§4.2 Standard signals\n(all period 1/50 Hz)')
ax3.legend(fontsize=9)

# Parseval bar chart
P_vals_plot = {n: np.mean(x**2) for n,x in signals.items()}
axes[1][0].bar(range(len(P_vals_plot)),list(P_vals_plot.values()),
               color=plt.cm.tab10(np.linspace(0,0.9,len(P_vals_plot))),alpha=0.8,edgecolor='#333')
axes[1][0].set_xticks(range(len(P_vals_plot)))
axes[1][0].set_xticklabels([n[:10] for n in P_vals_plot.keys()],rotation=30,fontsize=8)
axes[1][0].set_ylabel('Average power'); axes[1][0].set_title('§4.2 Average power\nby signal type')

# Impulse response
ax5 = axes[1][1]
t_ir_ms = t_ir*1000
ax5.plot(t_ir_ms, h_rc*RC, 'royalblue', lw=2.5, label='h(t)=exp(-t/RC)/RC')
ax5.set_xlabel('t (ms)'); ax5.set_ylabel('h(t) (normalized)')
ax5.set_title('§4.4 RC filter impulse response\nh(t) = (1/RC)e^{-t/RC}')
ax5.legend(fontsize=9)

# Transfer function magnitude
ax6 = axes[1][2]
ax6.semilogx(f_H[1:], 20*np.log10(np.abs(H_jw[1:])+1e-15), 'royalblue', lw=2.5)
ax6.axvline(1000, color='red', ls='--', lw=1.5, label='f_c=1kHz')
ax6.axhline(-3,   color='gray',ls='--', lw=1.5, label='-3dB point')
ax6.set_xlabel('Frequency (Hz)'); ax6.set_ylabel('|H(f)| (dB)')
ax6.set_title('§4.4 RC LP filter H(f)\nBode magnitude')
ax6.legend(fontsize=9); ax6.set_xlim(10,fs/2)

plt.suptitle('§4: Average Power · Dirac Delta · PSD · Parseval · Impulse Response',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §5 💾 Memory Hierarchy — Cache Lines · Stride · Spatial/Temporal Locality

**Cache hierarchy** (typical x86-64, 2024):

| Level | Size | Latency | Bandwidth |
|-------|------|---------|-----------|
| L1 D-cache | 32–64 KB | 4 cycles | ~500 GB/s |
| L2 | 512 KB–1 MB | 12 cycles | ~200 GB/s |
| L3 | 8–32 MB | 40 cycles | ~60 GB/s |
| DRAM | 16–64 GB | 200 cycles | ~50 GB/s |

**Cache line:** 64 bytes = 8 doubles = 16 floats. Fetching one element loads the whole line.

**Stride-1 (row-major):** sequential access — every element is in the cache line.
**Stride-k:** every k-th element — misses every k/8 elements for doubles.

In [ ]:
# §5 Memory hierarchy — stride, locality, layout effects

import numpy as np
import time

# §5.1 Stride access benchmark
print('§5.1 Stride access pattern benchmark:')
N_mem   = 2**22   # 4M doubles = 32MB (exceeds L3)
arr_mem = np.random.randn(N_mem)
REPS    = 5

print(f'  Array size: {arr_mem.nbytes/1e6:.0f} MB')
print(f'  {"Stride":8s} {"Time(ms)":10s} {"GB/s":8s} {"Relative"}')
results_stride = {}
t_stride1 = None
for stride in [1, 2, 4, 8, 16, 32, 64, 128]:
    indices = np.arange(0, N_mem, stride)
    t0 = time.perf_counter()
    for _ in range(REPS):
        s = arr_mem[indices].sum()
    dt = (time.perf_counter()-t0)/REPS
    bw = len(indices)*8/dt/1e9
    if stride==1: t_stride1=dt
    rel = dt/t_stride1
    results_stride[stride] = (dt,bw,rel)
    print(f'  {stride:8d} {dt*1000:10.3f} {bw:8.2f} {rel:.2f}x slower')

# §5.2 Matrix transpose (cache blocking demo)
print('\n§5.2 Matrix transpose (cache blocking):')
M_size = 1024
A_mat  = np.random.randn(M_size, M_size)

def transpose_naive(A):
    B = np.empty_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            B[j,i] = A[i,j]
    return B

def transpose_blocked(A, block=32):
    B = np.empty_like(A)
    n = A.shape[0]
    for i in range(0,n,block):
        for j in range(0,n,block):
            B[j:j+block, i:i+block] = A[i:i+block, j:j+block].T
    return B

t0 = time.perf_counter()
for _ in range(20): B_np = A_mat.T.copy()
t_np = (time.perf_counter()-t0)/20

t0 = time.perf_counter()
for _ in range(5): B_blk = transpose_blocked(A_mat, block=64)
t_blk = (time.perf_counter()-t0)/5

print(f'  numpy .T:      {t_np*1000:.3f} ms  {A_mat.nbytes*2/t_np/1e9:.2f} GB/s')
print(f'  blocked(b=64): {t_blk*1000:.3f} ms  (Python loop overhead, but cache-friendly)')
print(f'  numpy uses highly optimized BLAS -- blocked is the algorithmic idea')

# §5.3 C-order vs F-order sum
print('\n§5.3 Memory layout: C-order vs F-order:')
N2 = 2048
Ac = np.ascontiguousarray(np.random.randn(N2,N2))
Af = np.asfortranarray(Ac)

for axis,name_ax in [(0,'rows (axis=0)'),(1,'cols (axis=1)')]:
    t0 = time.perf_counter()
    for _ in range(50): Ac.sum(axis=axis)
    t_c = (time.perf_counter()-t0)/50*1000

    t0 = time.perf_counter()
    for _ in range(50): Af.sum(axis=axis)
    t_f = (time.perf_counter()-t0)/50*1000

    print(f'  sum({name_ax:12s}): C-order={t_c:.3f}ms  F-order={t_f:.3f}ms  ratio={t_f/t_c:.2f}')

# §5.4 Matrix multiply cache analysis
print('\n§5.4 Matmul FLOP/byte ratio (arithmetic intensity):')
for N_mm in [64, 128, 256, 512, 1024, 2048]:
    flops   = 2*N_mm**3   # 2N^3 FP ops
    bytes_r = 3*N_mm**2*8   # read A,B,C in double
    arith_i = flops/bytes_r
    cache_ok= arith_i > 40   # L3 bandwidth limited above ~40 FLOP/byte
    print(f'  N={N_mm:5d}: {flops/1e9:.2f} GFLOP  AI={arith_i:.1f} FLOP/byte  '
          f'{"compute-bound" if cache_ok else "memory-bound":15s}')

# §5.5 Roofline model
print('\n§5.5 Roofline model:')
peak_gflops = 200.0    # e.g. A100 tensor core peak
peak_bw_gbs = 2000.0   # HBM2e bandwidth GB/s
ridge_point = peak_gflops/peak_bw_gbs   # FLOP/byte at knee
print(f'  Peak compute: {peak_gflops} GFLOP/s')
print(f'  Peak bandwidth: {peak_bw_gbs} GB/s')
print(f'  Ridge point: {ridge_point:.2f} FLOP/byte')

ai_range = np.logspace(-2, 4, 200)
perf_roof= np.minimum(peak_gflops, ai_range*peak_bw_gbs)

fig, axes = plt.subplots(1,3,figsize=(15,5))
# Stride bandwidth
strides_v = list(results_stride.keys())
bw_vals   = [v[1] for v in results_stride.values()]
axes[0].semilogx(strides_v, bw_vals, 'o-', color='royalblue', lw=2.5, ms=8)
axes[0].set_xlabel('Stride'); axes[0].set_ylabel('Bandwidth (GB/s)')
axes[0].set_title('§5.1 Memory bandwidth\nvs access stride')
axes[0].axvline(8,color='red',ls='--',lw=1.5,label='Cache line boundary (8 doubles)')
axes[0].legend(fontsize=8)

# Roofline
axes[1].loglog(ai_range, perf_roof, 'royalblue', lw=3, label='Roofline')
axes[1].axvline(ridge_point,color='red',ls='--',lw=2,label=f'Ridge {ridge_point:.1f} FLOP/B')
for N_mm,col in [(64,'red'),(256,'orange'),(1024,'green'),(2048,'blue')]:
    ai_mm = 2*N_mm**3/(3*N_mm**2*8)
    p_mm  = min(peak_gflops, ai_mm*peak_bw_gbs)
    axes[1].plot(ai_mm,p_mm,marker='*',ms=14,color=col,label=f'N={N_mm}')
axes[1].set_xlabel('Arithmetic intensity (FLOP/byte)')
axes[1].set_ylabel('Performance (GFLOP/s)')
axes[1].set_title('§5.5 Roofline model\nmatmul for various N')
axes[1].legend(fontsize=7)

# Cache hierarchy schematic
ax3 = axes[2]; ax3.axis('off')
ax3.set_title('§5 Cache hierarchy\n(memory mountain)', fontsize=10)
levels = [
    ('Registers\n~kB  1 cycle',    '#e74c3c', 0.90),
    ('L1 D-cache\n32-64KB  4 cy',  '#e67e22', 0.73),
    ('L2 cache\n512KB  12 cy',     '#f1c40f', 0.56),
    ('L3 cache\n8-32MB  40 cy',    '#27ae60', 0.39),
    ('DRAM\n16-64GB  200 cy',      '#3498db', 0.22),
    ('NVMe SSD\n1-4TB  10us',      '#9b59b6', 0.05),
]
for name,col,y_pos in levels:
    w = 0.55 + (0.90-y_pos)*0.5
    ax3.add_patch(mpatches.FancyBboxPatch((0.5-w/2,y_pos-0.06),w,0.11,
                                           boxstyle='round,pad=0.01',
                                           fc=col,ec='white',lw=1.5,alpha=0.85))
    ax3.text(0.5,y_pos,name,ha='center',va='center',fontsize=8.5,
             color='white',fontweight='bold')

plt.suptitle('§5: Memory Hierarchy — Stride Bandwidth · Cache Blocking · Roofline',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §6 🔢 6-Vectors · Matrix Multiply · FLOP Counting · Tiling

**4-vector (Lorentz):** $(ct, x, y, z)$ with metric $g_{\mu\nu} = \text{diag}(+,-,-,-)$

**6-vector (bivector / EM field tensor):** $F^{\mu\nu}$ has 6 independent components
— the electric and magnetic fields:
$$F^{\mu\nu} = \begin{pmatrix}0 & -E_x & -E_y & -E_z \\ E_x & 0 & -B_z & B_y \\ E_y & B_z & 0 & -B_x \\ E_z & -B_y & B_x & 0\end{pmatrix}$$

This is the covariant unification of $\vec{E}$ and $\vec{B}$ — a single 4×4 antisymmetric tensor.

**Matrix multiply tiling:** divide $N\times N$ matrices into $B\times B$ blocks.
Each block fits in L1/L2 cache → $O(N^3/B)$ cache misses vs $O(N^3)$ naive.

In [ ]:
# §6 6-vectors + matrix multiply

import numpy as np
import sympy as sp
import time

# §6.1 EM field tensor (the "6-vector")
print('§6.1 EM field tensor F^{mu nu} (the relativistic 6-vector):')
E_x,E_y,E_z = sp.symbols('E_x E_y E_z', real=True)
B_x,B_y,B_z = sp.symbols('B_x B_y B_z', real=True)

F_tensor = sp.Matrix([
    [0,    -E_x, -E_y, -E_z],
    [E_x,   0,   -B_z,  B_y],
    [E_y,   B_z,  0,   -B_x],
    [E_z,  -B_y,  B_x,  0  ],
])
print('  F^{mu nu} =')
sp.pprint(F_tensor, use_unicode=False)

# Lorentz invariants
g  = sp.diag(1,-1,-1,-1)   # metric
F_lower = g * F_tensor
F_sq    = F_tensor.T * F_lower
I1 = sp.trace(F_sq)
I1_s= sp.simplify(I1)
print(f'\n  First invariant Tr(F^2) = {I1_s}')
print(f'  = 2(B^2 - E^2/c^2)  -> Lorentz invariant (here c=1)')

# Dual tensor
G_tensor = sp.Matrix([
    [0,   -B_x, -B_y, -B_z],
    [B_x,  0,    E_z, -E_y],
    [B_y, -E_z,  0,    E_x],
    [B_z,  E_y, -E_x,  0  ],
])
I2 = sp.trace(F_tensor.T * g * G_tensor)
I2_s = sp.simplify(I2)
print(f'  Second invariant F*G = {I2_s}')
print(f'  = 4 E.B  -> also Lorentz invariant')

# §6.2 Lorentz boost of 4-vector
print('\n§6.2 Lorentz boost (4-vector):')
beta_val = 0.8
gamma_val= 1/np.sqrt(1-beta_val**2)
Lambda = np.array([
    [gamma_val,       -gamma_val*beta_val, 0, 0],
    [-gamma_val*beta_val, gamma_val,       0, 0],
    [0,                0,                 1, 0],
    [0,                0,                 0, 1],
])
# 4-momentum of a 1 GeV particle at rest
p4_rest = np.array([1.0, 0.0, 0.0, 0.0])   # (E/c, px, py, pz) in GeV/c units
p4_boosted = Lambda @ p4_rest
inv_mass_rest   = np.sqrt(p4_rest @ np.diag([1,-1,-1,-1]) @ p4_rest)
inv_mass_boosted= np.sqrt(p4_boosted @ np.diag([1,-1,-1,-1]) @ p4_boosted)
print(f'  beta={beta_val}  gamma={gamma_val:.4f}')
print(f'  p4 rest:    {p4_rest}  |p|={inv_mass_rest:.6f}')
print(f'  p4 boosted: {p4_boosted}  |p|={inv_mass_boosted:.6f}  (invariant mass preserved)')

# §6.3 Matrix multiply — FLOP counting, naive vs numpy
print('\n§6.3 Matrix multiply FLOP analysis:')
def matmul_naive(A,B):
    n,m = A.shape[0],B.shape[1]
    k   = A.shape[1]
    C   = np.zeros((n,m))
    for i in range(n):
        for j in range(m):
            for l_idx in range(k):
                C[i,j] += A[i,l_idx]*B[l_idx,j]
    return C

for N_mm in [64,128,256]:
    A = np.random.randn(N_mm,N_mm).astype(np.float32)
    B = np.random.randn(N_mm,N_mm).astype(np.float32)
    flops = 2*N_mm**3
    t0  = time.perf_counter()
    for _ in range(10): C_np = A @ B
    t_np = (time.perf_counter()-t0)/10
    gflops_np = flops/t_np/1e9
    # naive only for small N
    if N_mm <= 64:
        t0 = time.perf_counter()
        C_nv = matmul_naive(A[:N_mm,:N_mm].astype(float), B[:N_mm,:N_mm].astype(float))
        t_nv = time.perf_counter()-t0
        gflops_nv = flops/t_nv/1e9
        ok = np.allclose(C_np[:N_mm,:N_mm].astype(float), C_nv, atol=1e-4)
        print(f'  N={N_mm}: numpy {gflops_np:.1f} GFLOP/s  naive {gflops_nv:.3f} GFLOP/s  '
              f'speedup={gflops_np/gflops_nv:.0f}x  correct={ok}')
    else:
        print(f'  N={N_mm}: numpy {gflops_np:.1f} GFLOP/s  ({flops/1e9:.3f} GFLOP total)')

# §6.4 Tiled matrix multiply
def matmul_tiled(A, B, tile=32):
    N = A.shape[0]
    C = np.zeros((N,N))
    for i in range(0,N,tile):
        for j in range(0,N,tile):
            for k in range(0,N,tile):
                C[i:i+tile,j:j+tile] += A[i:i+tile,k:k+tile] @ B[k:k+tile,j:j+tile]
    return C

N_t = 256
A_t = np.random.randn(N_t,N_t)
B_t = np.random.randn(N_t,N_t)
t0  = time.perf_counter()
C_tiled = matmul_tiled(A_t, B_t, tile=32)
t_tile  = time.perf_counter()-t0
C_ref   = A_t @ B_t
print(f'\n§6.4 Tiled N={N_t} (tile=32):')
print(f'  Tiled time: {t_tile*1000:.1f}ms  numpy: <1ms  error={np.max(np.abs(C_tiled-C_ref)):.2e}')
print(f'  Tiling reduces cache misses O(N^3) -> O(N^3/B) where B=tile size')

# Visualization
fig, axes = plt.subplots(1,3,figsize=(15,5))
# F_tensor heatmap (numerical example)
E_ex = np.array([1.5,0.5,0.0])
B_ex = np.array([0.0,0.2,1.0])
F_num= np.array([[0,    -E_ex[0],-E_ex[1],-E_ex[2]],
                  [E_ex[0],  0,  -B_ex[2],  B_ex[1]],
                  [E_ex[1],B_ex[2],  0,    -B_ex[0]],
                  [E_ex[2],-B_ex[1],B_ex[0],  0    ]])
im0 = axes[0].imshow(F_num, cmap='RdBu_r', vmin=-2,vmax=2)
axes[0].set_title('§6.1 EM field tensor F^{mu nu}\n(E=(1.5,0.5,0), B=(0,0.2,1))')
for i in range(4):
    for j in range(4):
        axes[0].text(j,i,f'{F_num[i,j]:.1f}',ha='center',va='center',fontsize=10,fontweight='bold')
axes[0].set_xticks([0,1,2,3]); axes[0].set_xticklabels(['ct','x','y','z'])
axes[0].set_yticks([0,1,2,3]); axes[0].set_yticklabels(['ct','x','y','z'])
plt.colorbar(im0,ax=axes[0])

# Matmul speedup
N_range_mm = [32,64,128,256,512]
speedups = []
for N_mm in N_range_mm:
    flops = 2*N_mm**3
    A = np.random.randn(N_mm,N_mm).astype(np.float32)
    B = np.random.randn(N_mm,N_mm).astype(np.float32)
    t0 = time.perf_counter()
    for _ in range(20): _ = A@B
    t_np = (time.perf_counter()-t0)/20
    speedups.append(flops/t_np/1e9)
axes[1].plot(N_range_mm,speedups,'o-',color='royalblue',lw=2.5,ms=8)
axes[1].set_xlabel('Matrix size N'); axes[1].set_ylabel('GFLOP/s')
axes[1].set_title('§6.3 numpy matmul performance\nvs matrix size (BLAS DGEMM)')

# Tiling diagram
ax3 = axes[2]; ax3.axis('off')
ax3.set_title('§6.4 Cache-tiled matmul\n(blocked for L1/L2)', fontsize=10)
for i_b in range(4):
    for j_b in range(4):
        col_t = '#3498db' if (i_b+j_b)<3 else '#ecf0f1'
        ax3.add_patch(mpatches.Rectangle((j_b*0.22+0.05,0.75-i_b*0.17),
                                          0.20,0.15,fc=col_t,ec='#333',lw=1.5))
        if i_b+j_b == 2:
            ax3.text(j_b*0.22+0.15,0.75-i_b*0.17+0.075,'B\ntile',
                     ha='center',va='center',fontsize=8,fontweight='bold',color='white')
ax3.text(0.5,0.92,'NxN matrix A',ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
ax3.text(0.5,0.12,'Tile B fits in L1; 8x speedup over naive',ha='center',fontsize=9,color='gray')

plt.suptitle('§6: EM 6-Tensor · Lorentz Boost · Matmul FLOP · Cache Tiling',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §7 🔬 ASIC Matmul + Photonics — Systolic Array · MZI Mesh

**Systolic array** (Google TPU, Apple Neural Engine): $N\times N$ array of MAC units.
Each PE accumulates $C_{ij} += A_{ik} \cdot B_{kj}$ as data flows in from edges.
Throughput: $2N^3$ MACs in $3N-2$ cycles → $O(N^2)$ PEs, $O(N)$ time.

**Optical matrix multiply (MZI mesh):** unitary $N\times N$ matrix decomposed
into $N(N-1)/2$ Mach-Zehnder interferometers (Reck/Clements decomposition).
Each MZI implements a 2×2 rotation. Speed = $c/n$, power = zero (passive).

**Energy advantage:** photonic matmul energy $\propto 0$ (passive MZI) vs
ASIC matmul $\propto N^2 \cdot E_{MAC}$ (digital).

In [ ]:
# §7 ASIC matmul (systolic array) + MZI optical matmul

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# §7.1 Systolic array simulation (N=4)
print('§7.1 Systolic array matrix multiply (N=4):')

def systolic_matmul(A, B):
    '''Simulate weight-stationary systolic array.'''
    N = A.shape[0]
    # Each PE holds one weight (B element), accumulates
    C = np.zeros((N,N))
    # Stagger inputs (skew A rows by their row index)
    for cycle in range(3*N-2):
        for i in range(N):
            for j in range(N):
                k = cycle - i - j   # which K index arrives at PE(i,j) this cycle
                if 0 <= k < N:
                    C[i,j] += A[i,k] * B[k,j]
    return C

N_sys = 4
A_sys = np.random.randint(1,5,(N_sys,N_sys)).astype(float)
B_sys = np.random.randint(1,5,(N_sys,N_sys)).astype(float)
C_sys = systolic_matmul(A_sys, B_sys)
C_ref = A_sys @ B_sys
print(f'  N={N_sys}: systolic == numpy: {np.allclose(C_sys,C_ref)}')
print(f'  Cycles: {3*N_sys-2}  PEs: {N_sys**2}  MACs: {2*N_sys**3}')
print(f'  MACs/cycle: {2*N_sys**3/(3*N_sys-2):.1f}  (ideal={N_sys**2})')

# Performance model
print('\n  Systolic array performance model:')
freq_GHz = 1.0   # 1 GHz systolic array
E_mac    = 0.1e-12   # 100 fJ per MAC (INT8)
for N_sa in [8,16,32,64,128]:
    n_PEs    = N_sa**2
    macs_cyc = N_sa**2
    tops     = macs_cyc * freq_GHz * 1e9 * 2 / 1e12   # TOPS (MACs = 2 FLOPs)
    power_W  = n_PEs * freq_GHz*1e9 * E_mac
    eff      = tops/power_W   # TOPS/W
    print(f'  N={N_sa:4d}: {n_PEs:6d} PEs  {tops:.1f} TOPS  {power_W:.2f}W  {eff:.1f} TOPS/W')

# §7.2 MZI mesh (Clements decomposition)
print('\n§7.2 MZI photonic matrix multiply:')

def mzi_2x2(theta, phi):
    '''2x2 MZI transfer matrix.'''
    return np.array([
        [np.exp(1j*phi)*np.cos(theta), -np.sin(theta)],
        [np.exp(1j*phi)*np.sin(theta),  np.cos(theta)]
    ])

def clements_decompose(U):
    '''Simplified: decompose NxN unitary into MZI product (Clements scheme).'''
    N = U.shape[0]
    V = U.copy().astype(complex)
    mzis = []
    # Forward pass: null upper triangle column by column
    for col in range(N-1):
        for row in range(N-1, col, -1):
            # Find theta to null V[row,col]
            if abs(V[row-1,col]) < 1e-12:
                theta = np.pi/2
            else:
                theta = np.arctan2(abs(V[row,col]),abs(V[row-1,col]))
            phi   = np.angle(V[row,col]) - np.angle(V[row-1,col])
            # Apply MZI from left
            M = np.eye(N, dtype=complex)
            M2 = mzi_2x2(theta, phi)
            M[row-1:row+1, row-1:row+1] = M2
            V = M @ V
            mzis.append((row-1, theta, phi))
    return mzis, V   # V should be diagonal

N_mzi = 4
# Random unitary (QR decomposition)
Q,_ = np.linalg.qr(np.random.randn(N_mzi,N_mzi)+1j*np.random.randn(N_mzi,N_mzi))
mzis, D_res = clements_decompose(Q)
# Check unitarity preserved
print(f'  N={N_mzi} unitary matrix:')
print(f'  Number of MZIs needed: {len(mzis)}  (theory: N(N-1)/2={N_mzi*(N_mzi-1)//2})')
print(f'  Residual off-diagonal: {np.max(np.abs(D_res - np.diag(np.diag(D_res)))):.2e}')

# Energy comparison: digital vs photonic
print('\n§7.3 Energy comparison:')
N_compare = 256
macs      = 2*N_compare**3
E_dig_J   = macs * 0.1e-12   # 100 fJ/MAC digital INT8
E_opt_J   = N_compare * 1e-15 * N_compare  # ~1 fJ/element * N activations * N wavelengths
print(f'  N={N_compare} matmul ({macs/1e9:.2f} GMAC):')
print(f'  Digital ASIC: {E_dig_J*1e9:.2f} nJ  (100 fJ/MAC)')
print(f'  Photonic MZI: ~{E_opt_J*1e9:.3f} nJ  (passive propagation)')
print(f'  Energy advantage: ~{E_dig_J/E_opt_J:.0f}x  (but bandwidth limited by ADC/DAC)')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(15,5))

# Systolic array diagram
ax1 = axes[0]
ax1.axis('off')
ax1.set_title('§7.1 Systolic array (N=4)\nweight-stationary', fontsize=10)
for i in range(4):
    for j in range(4):
        col = '#3498db' if (i+j)%2==0 else '#2980b9'
        ax1.add_patch(mpatches.FancyBboxPatch((j*0.22+0.04, 0.72-i*0.19),
                                               0.18,0.15,
                                               boxstyle='round,pad=0.01',
                                               fc=col,ec='white',lw=1.5))
        ax1.text(j*0.22+0.13,0.72-i*0.19+0.075,
                 f'MAC\n({i},{j})',ha='center',va='center',
                 fontsize=7,color='white',fontweight='bold')
# Data flow arrows
for i in range(4):
    ax1.annotate('',xy=(0.04,0.72-i*0.19+0.075),xytext=(0.0,0.72-i*0.19+0.075),
                 arrowprops=dict(arrowstyle='-|>',color='red',lw=1.5))
    ax1.text(-0.01,0.72-i*0.19+0.075,f'A[{i},:]',fontsize=7,ha='right',color='red')
for j in range(4):
    ax1.annotate('',xy=(j*0.22+0.13,0.89),xytext=(j*0.22+0.13,0.93),
                 arrowprops=dict(arrowstyle='-|>',color='green',lw=1.5))
    ax1.text(j*0.22+0.13,0.95,f'B[:,{j}]',fontsize=7,ha='center',color='green')

# MZI mesh
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('§7.2 MZI mesh (N=4)\nClements decomposition', fontsize=10)
col_wg  = '#2c3e50'
# Draw waveguides
for row in range(4):
    y = 0.2+row*0.2
    ax2.plot([0.05,0.95],[y,y],color=col_wg,lw=2,zorder=1)
# Draw MZIs
mzi_positions = [(0.20,0),(0.20,1),(0.20,2),(0.40,1),(0.40,2),(0.60,0)]
for (x_mzi,row_mzi) in mzi_positions:
    y1 = 0.2+row_mzi*0.2; y2 = y1+0.2
    ax2.add_patch(mpatches.FancyBboxPatch((x_mzi-0.05,y1-0.03),0.10,y2-y1+0.06,
                                           boxstyle='round,pad=0.01',
                                           fc='#e74c3c',ec='white',lw=1.5,alpha=0.8))
    ax2.text(x_mzi,y1+0.1,'MZI',ha='center',va='center',fontsize=7,color='white',fontweight='bold')
for i_in in range(4):
    ax2.text(0.02,0.2+i_in*0.2,f'in{i_in}',fontsize=8,va='center',ha='left',color='#2c3e50')
    ax2.text(0.97,0.2+i_in*0.2,f'out{i_in}',fontsize=8,va='center',ha='right',color='#27ae60')

# Energy comparison bar chart
ax3 = axes[2]
systems = ['NVIDIA A100\n(FP16)', 'Google TPU v4\n(bfloat16)', 'Apple ANE\n(INT8)',
           'Lightmatter\n(photonic)', 'MZI mesh\n(theoretical)']
tops_vals = [312, 275, 38, 100, 1000]
topspw    = [0.6, 1.1, 30, 100, 1e4]  # TOPS/W
colors_a  = ['#3498db','#27ae60','#9b59b6','#e67e22','#e74c3c']
bars = ax3.bar(range(len(systems)), topspw, color=colors_a, alpha=0.85, edgecolor='#333')
ax3.set_yscale('log')
ax3.set_xticks(range(len(systems)))
ax3.set_xticklabels(systems,fontsize=7,rotation=15)
ax3.set_ylabel('TOPS/W (log scale)')
ax3.set_title('§7.3 AI accelerator efficiency\nphotonic orders of magnitude better')
for bar,v in zip(bars,topspw):
    ax3.text(bar.get_x()+bar.get_width()/2, v*1.3, f'{v}',
             ha='center',va='bottom',fontsize=8,fontweight='bold')

plt.suptitle('§7: Systolic Array · MZI Photonic Matmul · ASIC Energy Comparison',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §8 ⚡ Early Electromagnetism — Coulomb → Gauss → Faraday → Ampere → Maxwell

**Historical chain:**

1. **Coulomb (1785):** $F = kq_1q_2/r^2$ → action at a distance
2. **Gauss (1835):** $\oint \vec{E}\cdot d\vec{A} = Q_{enc}/\varepsilon_0$ → field concept
3. **Faraday (1831):** $\mathcal{E} = -d\Phi_B/dt$ → changing B creates E
4. **Ampere (1826):** $\oint \vec{B}\cdot d\vec{l} = \mu_0 I_{enc}$ → current creates B
5. **Maxwell (1865):** adds displacement current $\mu_0\varepsilon_0\partial\vec{E}/\partial t$ →
   predicts EM waves at $c = 1/\sqrt{\mu_0\varepsilon_0} = 3\times10^8$ m/s

In [ ]:
# §8 Early electromagnetism — from Coulomb to Maxwell

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# §8.1 Coulomb's law + superposition
print('§8.1 Coulomb force + electric field:')
k_e  = 8.99e9   # N m^2/C^2
eps0 = 8.85e-12  # F/m
mu0  = 4*np.pi*1e-7   # H/m
c_em = 1/np.sqrt(mu0*eps0)
print(f'  c = 1/sqrt(mu0*eps0) = {c_em:.5e} m/s  (Maxwell prediction)')
print(f'  Actual c = 2.99792e8 m/s  -> Maxwell was RIGHT')

# E-field from a point charge via Coulomb superposition
def E_field_grid(charges, positions, X, Y):
    '''Compute E-field on 2D grid from list of point charges.'''
    Ex = np.zeros_like(X)
    Ey = np.zeros_like(Y)
    for q, (x0,y0) in zip(charges,positions):
        dx = X-x0; dy = Y-y0
        r2 = dx**2+dy**2+1e-20
        Ex += k_e*q*dx/r2**1.5
        Ey += k_e*q*dy/r2**1.5
    return Ex, Ey

# §8.2 Gauss's law: verify Q_enc = eps0 * surface integral
print('\n§8.2 Gauss\'s law verification:')
# Charge at origin: q = 1 nC
q_test = 1e-9   # C
R_surf = 0.1    # 0.1 m sphere surface
# Surface integral of E dot dA over sphere = q/eps0
# E = kq/R^2 (uniform on sphere surface)
E_surf = k_e * q_test / R_surf**2
Phi_E  = E_surf * 4*np.pi*R_surf**2   # E * 4piR^2
print(f'  q = {q_test*1e9:.1f} nC  R = {R_surf} m')
print(f'  |E| on surface = {E_surf:.4e} V/m')
print(f'  Flux Phi_E = {Phi_E:.4e} V m')
print(f'  q/eps0     = {q_test/eps0:.4e} V m  -> Gauss satisfied: {abs(Phi_E-q_test/eps0)/abs(q_test/eps0) < 1e-10}')

# §8.3 Faraday's law: EMF from changing flux
print('\n§8.3 Faraday\'s law:')
t_sym, omega_sym, B0, A_sym = sp.symbols('t omega B_0 A', positive=True)
B_t = B0 * sp.cos(omega_sym * t_sym)   # time-varying B
Phi_B = A_sym * B_t                    # flux through area A
EMF  = -sp.diff(Phi_B, t_sym)
print(f'  B(t) = B0 cos(wt)')
print(f'  Phi(t) = A*B0*cos(wt)')
print(f'  EMF = -dPhi/dt = {EMF}')
print(f'  EMF amplitude = omega*A*B0  (proportional to frequency)')

# §8.4 Ampere's law + displacement current
print('\n§8.4 Displacement current (Maxwell correction):')
f_gap   = 1e6    # 1 MHz
omega_g = 2*np.pi*f_gap
C_cap   = 1e-9   # 1 nF capacitor
V_amp   = 1.0    # 1 V amplitude
I_cond  = V_amp * omega_g * C_cap   # conduction current
I_disp  = I_cond   # displacement current equals conduction current in capacitor
print(f'  Capacitor: C={C_cap*1e9:.0f}nF  f={f_gap/1e6:.0f}MHz  V={V_amp}V')
print(f'  Conduction current:   {I_cond*1000:.4f} mA')
print(f'  Displacement current: {I_disp*1000:.4f} mA  (eps0 * dE/dt * A)')
print(f'  Maxwell: without J_disp, Ampere\'s law violates continuity equation')

# §8.5 Wave equation from Maxwell
print('\n§8.5 Wave equation from Maxwell:')
x_sym, t_sym2, c_sym = sp.symbols('x t c', positive=True)
E_sym = sp.Function('E')(x_sym, t_sym2)
# Wave equation: d^2E/dx^2 = 1/c^2 * d^2E/dt^2
wave_eq = sp.Eq(sp.diff(E_sym,x_sym,2), (1/c_sym**2)*sp.diff(E_sym,t_sym2,2))
print(f'  {wave_eq}')
# Plane wave solution
k_sym, omega_w = sp.symbols('k omega', positive=True)
E_plane = sp.exp(sp.I*(k_sym*x_sym - omega_w*t_sym2))
lhs = sp.diff(E_plane,x_sym,2)
rhs = (1/c_sym**2)*sp.diff(E_plane,t_sym2,2)
dispersion = sp.simplify(lhs - rhs)
print(f'  Plane wave E=exp(i(kx-wt)) substitution -> {sp.collect(dispersion, E_plane)}')
print(f'  Dispersion: k^2 = omega^2/c^2  ->  omega = c*k  (linear, non-dispersive)')

# §8.6 Poynting vector
print('\n§8.6 Poynting vector (EM power flux):')
E0_em = 1000.0   # V/m (laser field)
H0_em = E0_em / (mu0*c_em)   # H = E/Z0 for plane wave
Z0    = np.sqrt(mu0/eps0)    # free space impedance ~377 ohm
S_avg = 0.5 * E0_em * H0_em  # time-averaged Poynting magnitude
I_irr = 0.5 * E0_em**2 / Z0  # irradiance W/m^2
print(f'  E0 = {E0_em:.0f} V/m  Z0 = {Z0:.1f} ohm')
print(f'  <S> = E0^2/(2*Z0) = {I_irr:.2f} W/m^2')
print(f'  1 sun = 1000 W/m^2  -> E0 = {np.sqrt(2*1000*Z0):.1f} V/m')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# E-field from dipole
Ng = 50
x_g = np.linspace(-0.3,0.3,Ng); y_g = np.linspace(-0.3,0.3,Ng)
Xg,Yg = np.meshgrid(x_g,y_g)
charges_d = [+1e-9, -1e-9]
pos_d     = [(0.05,0),(-0.05,0)]
Ex,Ey = E_field_grid(charges_d, pos_d, Xg, Yg)
E_mag = np.sqrt(Ex**2+Ey**2)
E_mag_clipped = np.clip(E_mag, 0, 5e5)
im1 = axes[0][0].streamplot(x_g,y_g,Ex,Ey,color=np.log10(E_mag+1),
                              cmap='hot',density=2,linewidth=1.5)
axes[0][0].plot([0.05,- 0.05],[0,0],'b+',ms=12,mew=2.5)
axes[0][0].plot([-0.05,-0.05],[0,0],'r_',ms=12,mew=2.5)
axes[0][0].set_title('§8.1 Coulomb + superposition\nDipole E-field lines')
axes[0][0].set_xlabel('x (m)'); axes[0][0].set_ylabel('y (m)')

# Gauss law surface integral visualization
theta_g2 = np.linspace(0,2*np.pi,300)
r_vals = [0.05,0.10,0.20]
for r_v,col in zip(r_vals,['red','orange','blue']):
    E_r = k_e*q_test/r_v**2
    axes[0][1].add_patch(plt.Circle((0,0),r_v,fill=False,color=col,lw=2,
                                     label=f'r={r_v}m flux={E_r*4*np.pi*r_v**2:.2e}'))
axes[0][1].plot(0,0,'k*',ms=15,label='+q charge')
axes[0][1].set_xlim(-0.25,0.25); axes[0][1].set_ylim(-0.25,0.25)
axes[0][1].set_aspect('equal'); axes[0][1].legend(fontsize=7)
axes[0][1].set_title("§8.2 Gauss's law\nflux constant for any surface")

# Faraday EMF
t_far = np.linspace(0,4*np.pi,500)
B_far = np.cos(t_far)
emf_far = np.sin(t_far)   # -d/dt cos(t) = sin(t)
axes[0][2].plot(t_far,B_far,'royalblue',lw=2.5,label='B(t)=cos(t)')
axes[0][2].plot(t_far,emf_far,'red',lw=2.5,label='EMF=-dPhi/dt=sin(t)')
axes[0][2].axhline(0,color='gray',lw=0.5)
axes[0][2].set_xlabel('t (rad)'); axes[0][2].legend(fontsize=9)
axes[0][2].set_title("§8.3 Faraday's law\nEMF 90 degrees ahead of B")

# Plane wave E and B
x_wave = np.linspace(0, 4*np.pi, 500)
E_wave = np.sin(x_wave)
B_wave = np.sin(x_wave)
axes[1][0].plot(x_wave, E_wave, 'royalblue', lw=2.5, label='E-field')
axes[1][0].plot(x_wave, B_wave*0.5,'red',lw=2.5,ls='--',label='B-field (x0.5)')
axes[1][0].set_xlabel('x (kz)'); axes[1][0].legend(fontsize=9)
axes[1][0].set_title('§8.5 Plane EM wave\nc=1/sqrt(mu0*eps0)')

# Maxwell equations summary
ax_max = axes[1][1]
ax_max.axis('off')
ax_max.set_title('§8 Maxwell\'s equations\n(differential form)', fontsize=10)
maxwells = [
    (r'$\nabla\cdot\vec{E} = \rho/\varepsilon_0$',     'Gauss (E)',  '#3498db'),
    (r'$\nabla\cdot\vec{B} = 0$',                        'No monopoles','#27ae60'),
    (r'$\nabla\times\vec{E} = -\partial\vec{B}/\partial t$','Faraday',  '#e67e22'),
    (r'$\nabla\times\vec{B} = \mu_0\vec{J}+\mu_0\varepsilon_0\partial\vec{E}/\partial t$',
     'Ampere-Maxwell','#e74c3c'),
]
for i,(eq,name,col) in enumerate(maxwells):
    y_i = 0.80-i*0.20
    ax_max.add_patch(mpatches.FancyBboxPatch((0.02,y_i-0.08),0.95,0.14,
                                              boxstyle='round,pad=0.01',
                                              fc=col,ec='white',lw=1.5,alpha=0.85))
    ax_max.text(0.50,y_i,f'{eq}\n({name})',ha='center',va='center',
                fontsize=10,color='white',fontweight='bold')

# Poynting: irradiance vs field amplitude
E0_range = np.logspace(1,6,200)
I_range  = 0.5*E0_range**2/Z0
axes[1][2].loglog(E0_range, I_range, 'royalblue', lw=2.5)
axes[1][2].axhline(1000,color='orange',ls='--',lw=1.5,label='1 sun (1kW/m^2)')
axes[1][2].axhline(1e12,color='red',ls='--',lw=1.5,label='Laser cutting ~TW/m^2')
axes[1][2].set_xlabel('E0 (V/m)'); axes[1][2].set_ylabel('Irradiance (W/m^2)')
axes[1][2].set_title('§8.6 Poynting vector\nirradiance vs E-field amplitude')
axes[1][2].legend(fontsize=8)

plt.suptitle('§8: Coulomb to Maxwell — E-field · Gauss · Faraday · Wave Equation · Poynting',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §9 🔢 Integer Arithmetic — Python Big Int · Montgomery · Modular Exponentiation

**Python `int`:** arbitrary precision — stored as array of 30-bit "digits".
Multiplication of $n$-bit numbers: $O(n^2)$ schoolbook, $O(n^{1.585})$ Karatsuba,
$O(n \log n \log\log n)$ Harvey-Hoeven FFT-based (Python 3.11+ for large N).

**Modular exponentiation** $a^e \mod m$ via square-and-multiply: $O(\log e)$ multiplications.

**Montgomery multiplication:** avoids expensive division in $a \cdot b \mod m$.
Transforms to Montgomery domain, multiplies, transforms back.
Used in RSA, ECC, AES key schedule.

**Why integers "took too long":** Python `int` arithmetic is slow for crypto-sized
numbers (2048-bit RSA) because it's pure Python overhead. Use `gmpy2` or `cryptography` lib.

In [ ]:
# §9 Integer arithmetic — big int, Montgomery, modular expo

import time, math, sys

# §9.1 Python big int representation
print('§9.1 Python big int sizes:')
for bits in [8,32,64,128,256,1024,2048,4096]:
    n  = (1 << bits) - 1   # 2^bits - 1
    nb = sys.getsizeof(n)
    print(f'  {bits:5d}-bit: sys.getsizeof={nb:4d} bytes  digits={math.ceil(bits/30)} x 30-bit limbs')

# §9.2 Schoolbook vs Python big int multiply timing
print('\n§9.2 Multiplication timing:')
for bits in [64, 128, 256, 512, 1024, 2048]:
    import random
    random.seed(42)
    a = random.getrandbits(bits)
    b = random.getrandbits(bits)
    REPS = max(10, 10000//bits)
    t0 = time.perf_counter()
    for _ in range(REPS): c = a*b
    dt = (time.perf_counter()-t0)/REPS
    print(f'  {bits:5d}-bit mul: {dt*1e9:.1f} ns  ({REPS} reps)')

# §9.3 Square-and-multiply modular exponentiation
def powmod_manual(base, exp, mod):
    '''Square-and-multiply (right-to-left binary method).'''
    result = 1
    base   = base % mod
    while exp > 0:
        if exp & 1:         # if current bit is 1
            result = (result * base) % mod
        exp  >>= 1          # shift right (next bit)
        base  = (base * base) % mod
    return result

# Test
print('\n§9.3 Modular exponentiation:')
for bits_e, bits_m in [(64,128),(256,512),(512,1024),(1024,2048)]:
    import random
    random.seed(7)
    base_v= random.getrandbits(bits_m//2)
    exp_v = random.getrandbits(bits_e)
    mod_v = random.getrandbits(bits_m) | 1   # odd modulus

    t0  = time.perf_counter()
    r1  = powmod_manual(base_v, exp_v, mod_v)
    t_man = time.perf_counter()-t0

    t0  = time.perf_counter()
    r2  = pow(base_v, exp_v, mod_v)   # Python built-in (C implementation)
    t_py= time.perf_counter()-t0

    match = (r1 == r2)
    print(f'  e={bits_e:5d}b m={bits_m:5d}b: manual={t_man*1000:.3f}ms  '
          f'pow()={t_py*1000:.4f}ms  speedup={t_man/max(t_py,1e-9):.0f}x  match={match}')

# §9.4 Montgomery multiplication (demo)
def montgomery_multiply(a, b, n, R, R_inv, n_prime):
    '''Montgomery multiplication: computes a*b*R^{-1} mod n.'''
    T   = a * b
    m   = (T * n_prime) % R   # reduction factor
    t   = (T + m*n) // R
    if t >= n:
        t -= n
    return t

print('\n§9.4 Montgomery multiplication demo (small numbers):')
n_mg   = 97      # odd modulus
R_mg   = 128     # R = 2^7 > n
# Find n_prime: n*n_prime = -1 mod R
n_prime_mg = pow(-n_mg, -1, R_mg)   # requires Python 3.8+
R_inv_mg   = pow(R_mg, -1, n_mg)

# Test: Montgomery(a*R mod n, b*R mod n) should give a*b*R mod n
a_mg, b_mg = 23, 41
# Convert to Montgomery domain
a_mon = (a_mg * R_mg) % n_mg
b_mon = (b_mg * R_mg) % n_mg
# Montgomery multiply
c_mon = montgomery_multiply(a_mon, b_mon, n_mg, R_mg, n_prime_mg, n_mg)
# Convert back
c_val = (c_mon * R_inv_mg) % n_mg
expected_mg = (a_mg * b_mg) % n_mg
print(f'  n={n_mg}  R={R_mg}  n_prime={n_prime_mg}')
print(f'  {a_mg} * {b_mg} mod {n_mg} = {expected_mg}')
print(f'  Montgomery result: {c_val}  match={c_val==expected_mg}')

# §9.5 GCD, extended Euclidean, modular inverse
print('\n§9.5 Extended Euclidean algorithm:')
def extended_gcd(a, b):
    if b == 0: return a, 1, 0
    g,x,y = extended_gcd(b, a%b)
    return g, y, x - (a//b)*y

def modinv(a, m):
    g,x,_ = extended_gcd(a%m, m)
    if g != 1: raise ValueError(f'No inverse: gcd({a},{m})={g}')
    return x%m

test_pairs = [(3,7),(17,3120),(65537,3233),(123456789,1000000007)]
for a_e,m_e in test_pairs:
    if math.gcd(a_e,m_e)==1:
        inv_e  = modinv(a_e,m_e)
        check  = (a_e*inv_e)%m_e
        py_inv = pow(a_e,-1,m_e)
        print(f'  {a_e}^{{-1}} mod {m_e} = {inv_e}  check={check}  py_pow={py_inv}  match={inv_e==py_inv}')

# §9.6 RSA mini-demo (educational, small primes)
print('\n§9.6 RSA mini-demo:')
p_rsa, q_rsa = 61, 53
n_rsa = p_rsa * q_rsa
phi_n = (p_rsa-1)*(q_rsa-1)
e_rsa = 17   # public exponent
d_rsa = modinv(e_rsa, phi_n)
print(f'  p={p_rsa}  q={q_rsa}  n={n_rsa}  phi(n)={phi_n}')
print(f'  e={e_rsa} (public)  d={d_rsa} (private)')
msg = 42
c_rsa = pow(msg, e_rsa, n_rsa)
m_rsa = pow(c_rsa, d_rsa, n_rsa)
print(f'  encrypt({msg}) = {c_rsa}  decrypt({c_rsa}) = {m_rsa}  match={m_rsa==msg}')

# Visualization
fig, axes = plt.subplots(1,3,figsize=(15,5))

# Big int multiplication time
bits_range = [64,128,256,512,1024,2048]
times_ms   = []
for bits in bits_range:
    import random
    random.seed(42)
    a_b = random.getrandbits(bits); b_b = random.getrandbits(bits)
    reps = max(5, 5000//bits)
    t0 = time.perf_counter()
    for _ in range(reps): _ = a_b*b_b
    times_ms.append((time.perf_counter()-t0)/reps*1e6)
axes[0].loglog(bits_range, times_ms, 'o-', color='royalblue', lw=2.5, ms=8)
# Fit slope
slope = np.polyfit(np.log(bits_range[2:]), np.log(times_ms[2:]),1)[0]
axes[0].set_xlabel('Bit length'); axes[0].set_ylabel('Time (us)')
axes[0].set_title(f'§9.2 Big int multiply\nslope={slope:.2f} (schoolbook ~2, Karatsuba ~1.58)')

# Square-and-multiply bit operations
exp_bits = 512
exp_ex   = (1<<exp_bits)-1   # all 1s: worst case
bits_arr = [(exp_ex >> i) & 1 for i in range(exp_bits)]
n_muls   = sum(bits_arr)     # number of multiply steps
n_sq     = exp_bits          # always exp_bits squarings
axes[1].bar(['Squarings','Multiplies'],[n_sq,n_muls],
            color=['royalblue','orange'],edgecolor='#333',alpha=0.85)
axes[1].set_title(f'§9.3 Square-and-multiply\n{exp_bits}-bit exponent ops')
axes[1].set_ylabel('Operations')
for rect,v in zip(axes[1].patches,[n_sq,n_muls]):
    axes[1].text(rect.get_x()+rect.get_width()/2,v+5,str(v),ha='center',fontsize=11)

# Modular inverse: Euclidean path
ax3 = axes[2]
ax3.axis('off')
ax3.set_title('§9.5 Extended Euclidean\n(Bezout coefficients)', fontsize=10)
steps = []
a_eu, b_eu = 65537, 3233*61*53   # example
a_tr, b_tr = a_eu, b_eu
while b_tr != 0:
    q_tr  = a_tr // b_tr
    steps.append((a_tr, b_tr, q_tr, a_tr%b_tr))
    a_tr, b_tr = b_tr, a_tr%b_tr
    if len(steps)>8: break
for i,(a_s,b_s,q_s,r_s) in enumerate(steps[:6]):
    y_i = 0.92-i*0.14
    col = '#27ae60' if r_s==0 else '#3498db'
    ax3.add_patch(mpatches.FancyBboxPatch((0.02,y_i-0.06),0.95,0.11,
                                           boxstyle='round,pad=0.01',fc=col,ec='white',lw=1.5,alpha=0.8))
    ax3.text(0.5,y_i,f'{a_s} = {q_s}*{b_s} + {r_s}',ha='center',va='center',
             fontsize=9,color='white',fontweight='bold')
ax3.text(0.5,0.04,'gcd found when remainder=0',ha='center',fontsize=8,color='gray')

plt.suptitle('§9: Integer Arithmetic — Python Big Int · Square-and-Multiply · Montgomery · RSA',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()